# Validate Routing Outputs

Run lightweight checks on generated routing features, sparse SLX edge lists, and yearly POI files.

In [ ]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

def find_project_dir(start: Path) -> Path:
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "README.md").exists() and (path / "ANAL").exists() and (path / "TOOLS").exists():
            return path
    raise FileNotFoundError("Could not find project root")


PROJECT_DIR = find_project_dir(Path.cwd())
ANAL_DATA = PROJECT_DIR / "ANAL" / "data"
ROUTING_DATA = ANAL_DATA / "routing"
OSM_DIR = PROJECT_DIR / "TOOLS" / "osm-data"
YEARS = range(2015, 2026)

ACTIVE_CELLS_PATH = ROUTING_DATA / "inputs" / "active_routing_cells_100m.parquet"
FIRMS_PATH = ANAL_DATA / "firms_assigned_100m.geoparquet"
PANEL_PATH = ANAL_DATA / "raster_quarter_panel_100m.parquet"
POPULATION_BACKCAST_PATH = ANAL_DATA / "population_backcast_100m_quarterly.parquet"
REPORT_ROOT = ROUTING_DATA / "reports"
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
START_YEAR = 2015
END_YEAR = 2025
SECTOR_IDS = [1, 2, 3, 4, 5, 6, 7]


In [ ]:
active_cells = pd.read_parquet(ACTIVE_CELLS_PATH)
active_grid_ids = set(active_cells["grid_id"])


def check(condition: bool, message: str) -> dict:
    return {"check": message, "status": "OK" if condition else "CHECK"}


def year_quarters(year: int) -> pd.PeriodIndex:
    return pd.period_range(f"{year}Q1", f"{year}Q4", freq="Q")


def accessibility_minutes() -> list[int]:
    return [15, 30]


def sector_stock_columns() -> list[str]:
    return [f"sparte_{sector_id}_active_firms_tminus1" for sector_id in SECTOR_IDS]


def accessibility_output_columns() -> list[str]:
    columns = []
    for minutes in accessibility_minutes():
        columns.extend([
            f"pop_access_{minutes}min",
            f"existing_firms_access_{minutes}min",
            *[f"sparte_{sector_id}_access_{minutes}min" for sector_id in SECTOR_IDS],
        ])
    return columns


def firm_accessibility_columns() -> list[str]:
    columns = []
    for minutes in accessibility_minutes():
        columns.extend([
            f"pop_access_{minutes}min",
            f"existing_firms_access_{minutes}min",
            f"same_sector_firms_access_{minutes}min",
        ])
    return columns


def load_yearly_accessibility_panel(year: int) -> pd.DataFrame:
    panel = pd.read_parquet(
        PANEL_PATH,
        columns=["grid_id", "year", "quarter", "period", "population_backcast", "active_firms_tminus1", *sector_stock_columns()],
        filters=[("year", "==", year)],
    )
    numeric_columns = ["population_backcast", "active_firms_tminus1", *sector_stock_columns()]
    for column in numeric_columns:
        panel[column] = pd.to_numeric(panel[column], errors="coerce").fillna(0.0).astype(float)
    panel["quarter"] = pd.to_numeric(panel["quarter"], errors="coerce").astype(int)
    panel["period"] = panel["period"].astype(str)
    return panel


_firm_source_cache: pd.DataFrame | None = None


def load_firm_source() -> pd.DataFrame:
    global _firm_source_cache
    if _firm_source_cache is not None:
        return _firm_source_cache
    firms = pd.read_parquet(FIRMS_PATH, columns=["firm_id", "grid_id_100m", "Sparte_ID", "founding_date", "exit_date"])
    firms = firms.dropna(subset=["firm_id", "grid_id_100m", "founding_date"]).copy()
    firms["founding_date"] = pd.to_datetime(firms["founding_date"], errors="coerce")
    firms["exit_date"] = pd.to_datetime(firms["exit_date"], errors="coerce")
    firms = firms.dropna(subset=["founding_date"]).copy()
    firms["exit_date_filled"] = firms["exit_date"].fillna(pd.Timestamp.max)
    firms["Sparte_ID_numeric"] = pd.to_numeric(firms["Sparte_ID"], errors="coerce").astype("Int64")
    _firm_source_cache = firms
    return _firm_source_cache


def build_expected_firm_quarter_panel_for_year(year: int) -> pd.DataFrame:
    firms = load_firm_source()
    records = []
    for period in year_quarters(year):
        quarter_end = period.end_time.normalize()
        previous_quarter_end = (period - 1).end_time.normalize()
        active_in_period = firms[(firms["founding_date"] <= quarter_end) & (firms["exit_date_filled"] > previous_quarter_end)].copy()
        if active_in_period.empty:
            continue
        active_in_period["year"] = int(period.year)
        active_in_period["quarter"] = int(period.quarter)
        active_in_period["period"] = str(period)
        active_in_period["included_in_lagged_stock"] = (
            (active_in_period["founding_date"] <= previous_quarter_end)
            & (active_in_period["exit_date_filled"] > previous_quarter_end)
        )
        records.append(active_in_period[["firm_id", "grid_id_100m", "Sparte_ID_numeric", "year", "quarter", "period", "included_in_lagged_stock"]])
    if not records:
        return pd.DataFrame(columns=["firm_id", "grid_id_100m", "Sparte_ID_numeric", "year", "quarter", "period", "included_in_lagged_stock"])
    return pd.concat(records, ignore_index=True)


def validate_pois(year: int) -> list[dict]:
    path = OSM_DIR / f"austria-{year}-pois.geoparquet"
    if not path.exists():
        return [check(False, f"{path.name} exists")]
    pois = gpd.read_parquet(path)
    expected_types = {"motorway_exit", "rail_station", "regional_centre", "urban_centre", "higher_education", "pt_stop"}
    pt_stops = pois[pois["poi_type"] == "pt_stop"].copy() if "poi_type" in pois.columns else gpd.GeoDataFrame()
    results = [
        check(len(pois) > 0, "POI file has rows"),
        check("poi_type" in pois.columns, "POI file has poi_type"),
        check(set(pois["poi_type"]).issubset(expected_types), "POI types are expected"),
        check(expected_types.issubset(set(pois["poi_type"])), "all routed POI types are present"),
        check("static_destination" in pois.columns, "POI file has static_destination flag"),
        check("pt_departures_weekday" in pois.columns, "POI file has pt_departures_weekday"),
        check(pois.geometry.notna().all(), "POI geometries are present"),
    ]
    if "pt_departures_weekday" in pois.columns and not pt_stops.empty:
        pt_departures = pd.to_numeric(pt_stops["pt_departures_weekday"], errors="coerce")
        results.extend([
            check(pt_departures.notna().all(), "PT stops have weekday departures"),
            check((pt_departures >= 0).all(), "PT stop weekday departures are non-negative"),
        ])
    return results


def validate_nearest_features(path: Path) -> list[dict]:
    if not path.exists():
        return [check(False, f"{path.name} exists")]
    nearest = pd.read_parquet(path)
    results = [
        check(len(nearest) == len(active_cells), "nearest feature row count matches active cells"),
        check("has_pt_stop_5min_walk" in nearest.columns, "nearest output has has_pt_stop_5min_walk"),
        check("pt_departures_5min_walk" in nearest.columns, "nearest output has pt_departures_5min_walk"),
    ]
    if "has_pt_stop_5min_walk" in nearest.columns:
        results.append(check(nearest["has_pt_stop_5min_walk"].notna().all(), "has_pt_stop_5min_walk has no nulls"))
    if "pt_departures_5min_walk" in nearest.columns:
        departures = pd.to_numeric(nearest["pt_departures_5min_walk"], errors="coerce")
        results.extend([
            check(departures.notna().all(), "pt_departures_5min_walk is numeric"),
            check((departures >= 0).all(), "pt_departures_5min_walk is non-negative"),
        ])
    required_columns = {"has_pt_stop_5min_walk", "pt_departures_5min_walk"}
    if required_columns.issubset(set(nearest.columns)):
        has_pt = nearest["has_pt_stop_5min_walk"].astype(bool)
        departures = pd.to_numeric(nearest["pt_departures_5min_walk"], errors="coerce").fillna(0.0)
        results.extend([
            check((departures[~has_pt] == 0.0).all(), "cells without PT stop have zero departures"),
            check((departures[has_pt] > 0).all(), "cells with PT stop have positive departures"),
        ])
    return results


def validate_population_accessibility(path: Path, year: int) -> list[dict]:
    if not path.exists():
        return [check(False, f"{path.name} exists")]
    potentials = pd.read_parquet(path)
    required_columns = {"grid_id", "year", "quarter", "period", "created_at", *accessibility_output_columns()}
    expected_rows = len(active_cells) * len(year_quarters(year))
    results = [
        check(len(potentials) == expected_rows, "population accessibility row count matches active cells times four quarters"),
        check(required_columns.issubset(set(potentials.columns)), "population accessibility has required columns"),
    ]
    if "created_at" in potentials.columns:
        results.append(check(potentials["created_at"].notna().all(), "population accessibility has created_at values"))
    if "year" in potentials.columns:
        results.append(check((pd.to_numeric(potentials["year"], errors="coerce") == year).all(), "population accessibility year column matches file year"))
    if {"quarter", "period"}.issubset(set(potentials.columns)):
        expected_periods = {str(quarter) for quarter in year_quarters(year)}
        results.extend([
            check(set(pd.to_numeric(potentials["quarter"], errors="coerce").dropna().astype(int).unique()) == {1, 2, 3, 4}, "population accessibility covers quarters 1-4"),
            check(set(potentials["period"].dropna().astype(str).unique()) == expected_periods, "population accessibility period labels match the year quarters"),
        ])
    if {"pop_access_15min", "pop_access_30min", "existing_firms_access_15min", "existing_firms_access_30min"}.issubset(set(potentials.columns)):
        pop_access_15 = pd.to_numeric(potentials["pop_access_15min"], errors="coerce")
        pop_access_30 = pd.to_numeric(potentials["pop_access_30min"], errors="coerce")
        existing_access_15 = pd.to_numeric(potentials["existing_firms_access_15min"], errors="coerce")
        existing_access_30 = pd.to_numeric(potentials["existing_firms_access_30min"], errors="coerce")
        results.extend([
            check(pop_access_15.notna().all(), "pop_access_15min is numeric"),
            check(pop_access_30.notna().all(), "pop_access_30min is numeric"),
            check((pop_access_15 >= 0).all(), "pop_access_15min is non-negative"),
            check((pop_access_30 >= 0).all(), "pop_access_30min is non-negative"),
            check((pop_access_30 >= pop_access_15).all(), "pop_access_30min is at least pop_access_15min"),
            check(existing_access_15.notna().all(), "existing_firms_access_15min is numeric"),
            check(existing_access_30.notna().all(), "existing_firms_access_30min is numeric"),
            check((existing_access_15 >= 0).all(), "existing_firms_access_15min is non-negative"),
            check((existing_access_30 >= 0).all(), "existing_firms_access_30min is non-negative"),
            check((existing_access_30 >= existing_access_15).all(), "existing_firms_access_30min is at least existing_firms_access_15min"),
        ])
        for sector_id in SECTOR_IDS:
            access_15 = pd.to_numeric(potentials[f"sparte_{sector_id}_access_15min"], errors="coerce")
            access_30 = pd.to_numeric(potentials[f"sparte_{sector_id}_access_30min"], errors="coerce")
            results.extend([
                check(access_15.notna().all(), f"sparte_{sector_id}_access_15min is numeric"),
                check(access_30.notna().all(), f"sparte_{sector_id}_access_30min is numeric"),
                check((access_15 >= 0).all(), f"sparte_{sector_id}_access_15min is non-negative"),
                check((access_30 >= 0).all(), f"sparte_{sector_id}_access_30min is non-negative"),
                check((access_30 >= access_15).all(), f"sparte_{sector_id}_access_30min is at least sparte_{sector_id}_access_15min"),
            ])
        sector_sum_15 = sum(pd.to_numeric(potentials[f"sparte_{sector_id}_access_15min"], errors="coerce").fillna(0.0) for sector_id in SECTOR_IDS)
        sector_sum_30 = sum(pd.to_numeric(potentials[f"sparte_{sector_id}_access_30min"], errors="coerce").fillna(0.0) for sector_id in SECTOR_IDS)
        results.extend([
            check(np.allclose(sector_sum_15, existing_access_15.fillna(0.0), atol=1e-6), "sector 15-minute accessibility sums to existing_firms_access_15min"),
            check(np.allclose(sector_sum_30, existing_access_30.fillna(0.0), atol=1e-6), "sector 30-minute accessibility sums to existing_firms_access_30min"),
        ])
        own_masses = active_cells[["grid_id"]].copy().merge(load_yearly_accessibility_panel(year), on="grid_id", how="left")
        merged = potentials.merge(own_masses, on=["grid_id", "year", "quarter", "period"], how="left")
        pop_access_15 = pd.to_numeric(merged["pop_access_15min"], errors="coerce").fillna(0.0)
        pop_access_30 = pd.to_numeric(merged["pop_access_30min"], errors="coerce").fillna(0.0)
        existing_access_15 = pd.to_numeric(merged["existing_firms_access_15min"], errors="coerce").fillna(0.0)
        existing_access_30 = pd.to_numeric(merged["existing_firms_access_30min"], errors="coerce").fillna(0.0)
        own_population_values = pd.to_numeric(merged["population_backcast"], errors="coerce").fillna(0.0)
        own_existing_values = pd.to_numeric(merged["active_firms_tminus1"], errors="coerce").fillna(0.0)
        results.extend([
            check((pop_access_15 >= own_population_values).all(), "pop_access_15min includes at least own population"),
            check((pop_access_30 >= own_population_values).all(), "pop_access_30min includes at least own population"),
            check((existing_access_15 >= own_existing_values).all(), "existing_firms_access_15min includes at least own lagged firm stock"),
            check((existing_access_30 >= own_existing_values).all(), "existing_firms_access_30min includes at least own lagged firm stock"),
        ])
    return results


def validate_firm_accessibility(path: Path, year: int) -> list[dict]:
    if not path.exists():
        return [check(False, f"{path.name} exists")]
    output = pd.read_parquet(path)
    expected_panel = build_expected_firm_quarter_panel_for_year(year)
    required_columns = {"firm_id", "grid_id_100m", "Sparte_ID", "year", "quarter", "period", "included_in_lagged_stock", *firm_accessibility_columns(), "created_at"}
    results = [
        check(len(output) == len(expected_panel), "firm accessibility row count matches the expected firm-quarter panel"),
        check(required_columns.issubset(set(output.columns)), "firm accessibility has required columns"),
    ]
    if {"firm_id", "year", "quarter"}.issubset(set(output.columns)):
        results.append(check(not output.duplicated(["firm_id", "year", "quarter"]).any(), "firm accessibility has one row per firm and quarter"))
    if {"created_at", "included_in_lagged_stock"}.issubset(set(output.columns)):
        results.extend([
            check(output["created_at"].notna().all(), "firm accessibility has created_at values"),
            check(output["included_in_lagged_stock"].notna().all(), "firm accessibility marks lagged-stock membership"),
        ])
    for column in firm_accessibility_columns():
        if column in output.columns:
            values = pd.to_numeric(output[column], errors="coerce")
            results.extend([
                check(values.notna().all(), f"{column} is numeric"),
                check((values >= 0).all(), f"{column} is non-negative"),
            ])
    if {"pop_access_15min", "pop_access_30min", "existing_firms_access_15min", "existing_firms_access_30min", "same_sector_firms_access_15min", "same_sector_firms_access_30min"}.issubset(set(output.columns)):
        pop_access_15 = pd.to_numeric(output["pop_access_15min"], errors="coerce").fillna(0.0)
        pop_access_30 = pd.to_numeric(output["pop_access_30min"], errors="coerce").fillna(0.0)
        existing_access_15 = pd.to_numeric(output["existing_firms_access_15min"], errors="coerce").fillna(0.0)
        existing_access_30 = pd.to_numeric(output["existing_firms_access_30min"], errors="coerce").fillna(0.0)
        same_sector_15 = pd.to_numeric(output["same_sector_firms_access_15min"], errors="coerce").fillna(0.0)
        same_sector_30 = pd.to_numeric(output["same_sector_firms_access_30min"], errors="coerce").fillna(0.0)
        results.extend([
            check((pop_access_30 >= pop_access_15).all(), "firm pop_access_30min is at least pop_access_15min"),
            check((existing_access_30 >= existing_access_15).all(), "firm existing_firms_access_30min is at least existing_firms_access_15min"),
            check((same_sector_15 <= existing_access_15).all(), "firm same-sector 15-minute access does not exceed total firm access"),
            check((same_sector_30 <= existing_access_30).all(), "firm same-sector 30-minute access does not exceed total firm access"),
        ])
        cell_accessibility = pd.read_parquet(ROUTING_DATA / "features" / str(year) / "accessibility_potentials_100m.parquet")
        cell_accessibility = cell_accessibility.rename(columns={"grid_id": "grid_id_100m"}).drop(columns=["created_at"], errors="ignore")
        merged = output.merge(
            cell_accessibility,
            on=["grid_id_100m", "year", "quarter", "period"],
            how="left",
            suffixes=("", "_cell"),
        )
        lagged_mask = merged["included_in_lagged_stock"].fillna(False).astype(bool)
        same_sector_expected_15 = pd.Series(0.0, index=merged.index)
        same_sector_expected_30 = pd.Series(0.0, index=merged.index)
        merged["Sparte_ID_numeric"] = pd.to_numeric(merged["Sparte_ID"], errors="coerce").astype("Int64")
        for sector_id in SECTOR_IDS:
            mask = merged["Sparte_ID_numeric"] == sector_id
            same_sector_expected_15.loc[mask] = pd.to_numeric(merged.loc[mask, f"sparte_{sector_id}_access_15min"], errors="coerce").fillna(0.0)
            same_sector_expected_30.loc[mask] = pd.to_numeric(merged.loc[mask, f"sparte_{sector_id}_access_30min"], errors="coerce").fillna(0.0)
        expected_existing_15 = pd.to_numeric(merged["existing_firms_access_15min_cell"], errors="coerce").fillna(0.0) - lagged_mask.astype(float)
        expected_existing_30 = pd.to_numeric(merged["existing_firms_access_30min_cell"], errors="coerce").fillna(0.0) - lagged_mask.astype(float)
        expected_same_sector_15 = same_sector_expected_15 - lagged_mask.astype(float)
        expected_same_sector_30 = same_sector_expected_30 - lagged_mask.astype(float)
        expected_existing_15 = expected_existing_15.clip(lower=0.0)
        expected_existing_30 = expected_existing_30.clip(lower=0.0)
        expected_same_sector_15 = expected_same_sector_15.clip(lower=0.0)
        expected_same_sector_30 = expected_same_sector_30.clip(lower=0.0)
        results.extend([
            check(np.allclose(existing_access_15, expected_existing_15, atol=1e-6), "firm existing_firms_access_15min matches cell access with conditional self-exclusion"),
            check(np.allclose(existing_access_30, expected_existing_30, atol=1e-6), "firm existing_firms_access_30min matches cell access with conditional self-exclusion"),
            check(np.allclose(same_sector_15, expected_same_sector_15, atol=1e-6), "firm same_sector_firms_access_15min matches sector access with conditional self-exclusion"),
            check(np.allclose(same_sector_30, expected_same_sector_30, atol=1e-6), "firm same_sector_firms_access_30min matches sector access with conditional self-exclusion"),
        ])
    return results


def validate_slx_edges(path: Path) -> list[dict]:
    if not path.exists():
        return [check(False, f"{path.name} exists")]
    edges = pd.read_parquet(path)
    row_sums = edges.groupby("origin_grid_id")["weight_rowstd"].sum()
    return [
        check((edges["origin_grid_id"] != edges["destination_grid_id"]).all(), "no self-neighbors"),
        check((edges["network_distance_m"] > 0).all(), "network distances are positive"),
        check((edges["network_distance_m"] <= 1000).all(), "main SLX cutoff is respected"),
        check((edges["weight_raw"] > 0).all(), "raw weights are positive"),
        check(np.allclose(row_sums, 1.0, atol=1e-6), "row-standardized weights sum to 1"),
        check(set(edges["origin_grid_id"]).issubset(active_grid_ids), "origin IDs exist in active cells"),
        check(set(edges["destination_grid_id"]).issubset(active_grid_ids), "destination IDs exist in active cells"),
    ]


In [ ]:
records = []
for year in YEARS:
    for result in validate_pois(year):
        records.append({"year": year, "product": "pois", **result})
    nearest_path = ROUTING_DATA / "features" / str(year) / "nearest_infrastructure_100m.parquet"
    if nearest_path.exists():
        for result in validate_nearest_features(nearest_path):
            records.append({"year": year, "product": "nearest_features", **result})
    potentials_path = ROUTING_DATA / "features" / str(year) / "accessibility_potentials_100m.parquet"
    if potentials_path.exists():
        for result in validate_population_accessibility(potentials_path, year):
            records.append({"year": year, "product": "population_accessibility", **result})
    firm_accessibility_path = ROUTING_DATA / "features" / str(year) / "firm_accessibility_quarter_100m.parquet"
    if firm_accessibility_path.exists():
        for result in validate_firm_accessibility(firm_accessibility_path, year):
            records.append({"year": year, "product": "firm_accessibility", **result})
    slx_path = ROUTING_DATA / "matrices" / str(year) / "W_local_drive_1km_hl500m_edges.parquet"
    if slx_path.exists():
        for result in validate_slx_edges(slx_path):
            records.append({"year": year, "product": "slx_edges", **result})

validation = pd.DataFrame(records)
validation.to_csv(REPORT_ROOT / "routing_validation_summary.csv", index=False)
validation
